# NumPy, Linear Systems, and ML Foundations

This notebook covers three foundational topics that underpin most of machine learning: efficient array manipulation with NumPy, solving systems of linear equations (including overdetermined systems via the pseudoinverse), and the taxonomy of machine learning paradigms. It closes with a conceptual treatment of regularization and why the optimal regularization coefficient must be selected on held-out data.

**Topics covered:**
- NumPy array creation, reshaping, and a reusable rounding utility
- Matrix inversion and the Vandermonde matrix
- Solving square and overdetermined linear systems in NumPy
- Moore-Penrose pseudoinverse and least-squares solutions
- Supervised, unsupervised, reinforcement, and rule-based paradigm classification
- Why λ* cannot be learned from training data alone
- Polynomial feature expansion for multivariate regression

---
## 1. NumPy Array Basics

### 1a. Array Creation and Reshaping

`np.arange` generates a flat integer sequence; `reshape` reinterprets its memory layout as a 2-D matrix without copying data. Row-major (C) order means elements fill left-to-right, top-to-bottom.

In [22]:
import numpy as np

np_array1 = np.arange(25).reshape(5, 5)
print("np_array1 =\n", np_array1)

np_array1 =
 [[ 0  1  2  3  4]
 [ 5  6  7  8  9]
 [10 11 12 13 14]
 [15 16 17 18 19]
 [20 21 22 23 24]]


### 1b. Linspace and a Reusable Rounding Utility

`np.linspace(start, stop, n)` produces `n` evenly-spaced values on the closed interval `[start, stop]`. The default float representation shows far more digits than are meaningful. A small wrapper around `np.round` restores legibility without losing the underlying precision of the array.

In [23]:
np_array2 = np.linspace(5.0, 10.0, 25)
print("np_array2 =\n", np_array2)

def matrix_round(arr, precision):
    """Return a copy of arr with all entries rounded to `precision` decimal places."""
    return np.round(arr, precision)

np_array2 =
 [ 5.          5.20833333  5.41666667  5.625       5.83333333  6.04166667
  6.25        6.45833333  6.66666667  6.875       7.08333333  7.29166667
  7.5         7.70833333  7.91666667  8.125       8.33333333  8.54166667
  8.75        8.95833333  9.16666667  9.375       9.58333333  9.79166667
 10.        ]


In [ ]:
# verify matrix_round on a sample matrix
test_matrix = np.array([
    [7.0,        7.42857143],
    [7.85714286, 8.28571429],
    [8.71428571, 9.14285714],
    [9.57142857, 10.0]
])

print("rounded matrix =\n", matrix_round(test_matrix, 3))

### 1c. Matrix Inversion and the Vandermonde Matrix

The matrix below is a **Vandermonde matrix** — each row is a geometric progression with ratio equal to the row index. Vandermonde matrices arise in polynomial interpolation, DFT, and Prony's method.

A square matrix $V$ is invertible iff $\det(V) \neq 0$. When it is, the products $V^{-1}V$ and $VV^{-1}$ both equal the **identity matrix** $I$ (up to floating-point rounding). NumPy generates identity matrices with `np.eye(n)` or `np.identity(n)`.

In [25]:
V = np.array([
    [1, 1,  1,   1,   1],
    [1, 2,  4,   8,  16],
    [1, 3,  9,  27,  81],
    [1, 4, 16,  64, 256],
    [1, 5, 25, 125, 625]
], dtype=float)

V_inv = np.linalg.inv(V)

print("V =\n", V)
print("\nV_inv (3 decimals) =\n", np.round(V_inv, 3))
print("\nV_inv @ V (3 decimals) =\n", np.round(V_inv @ V, 3))
print("\nV @ V_inv (3 decimals) =\n", np.round(V @ V_inv, 3))

V =
 [[  1.   1.   1.   1.   1.]
 [  1.   2.   4.   8.  16.]
 [  1.   3.   9.  27.  81.]
 [  1.   4.  16.  64. 256.]
 [  1.   5.  25. 125. 625.]]

V_inv (3 decimals) =
 [[  5.    -10.     10.     -5.      1.   ]
 [ -6.417  17.833 -19.5    10.167  -2.083]
 [  2.958  -9.833  12.25   -6.833   1.458]
 [ -0.583   2.167  -3.      1.833  -0.417]
 [  0.042  -0.167   0.25   -0.167   0.042]]

V_inv @ V (3 decimals) =
 [[ 1.  0.  0.  0.  0.]
 [-0.  1. -0. -0. -0.]
 [ 0.  0.  1.  0. -0.]
 [-0. -0. -0.  1. -0.]
 [ 0.  0.  0.  0.  1.]]

V @ V_inv (3 decimals) =
 [[ 1. -0.  0. -0.  0.]
 [-0.  1.  0. -0. -0.]
 [-0. -0.  1. -0. -0.]
 [-0. -0.  0.  1. -0.]
 [ 0. -0.  0. -0.  1.]]


Both products recover the identity matrix (the small negative zeros are floating-point artefacts from Gaussian elimination). NumPy provides `np.eye(n)` or `np.identity(n)` to generate identity matrices directly.

---
## 2. Solving Systems of Linear Equations

### 2a. Expressing a Linear System in Matrix Form

Any system of linear equations can be written compactly as $A\mathbf{b} = \mathbf{c}$, where $A$ is the coefficient matrix, $\mathbf{b}$ is the vector of unknowns, and $\mathbf{c}$ is the right-hand-side vector. For the system:

$$3x - y + 2z = 3$$
$$-y + z = 5$$
$$2x - 3z = -4$$

In [26]:
A = np.array([
    [3, -1,  2],
    [0, -1,  1],
    [2,  0, -3]
], dtype=float)

b = np.array([['x'], ['y'], ['z']])      # symbolic placeholder
c = np.array([3, 5, -4], dtype=float).reshape(3, 1)

print("A =\n", A)
print("\nb =\n", b)
print("\nc =\n", c)

A =
 [[ 3. -1.  2.]
 [ 0. -1.  1.]
 [ 2.  0. -3.]]

b =
 [['x']
 ['y']
 ['z']]

c =
 [[ 3.]
 [ 5.]
 [-4.]]


### 2b. Solving via Matrix Inverse

When $A$ is square and invertible, the unique solution is $\mathbf{b} = A^{-1}\mathbf{c}$. This follows directly from left-multiplying both sides of $A\mathbf{b} = \mathbf{c}$ by $A^{-1}$.

In [27]:
A_inv = np.linalg.inv(A)

print("A (3 decimals) =\n", np.round(A, 3))
print("\nA_inv (3 decimals) =\n", np.round(A_inv, 3))

A (3 decimals) =
 [[ 3. -1.  2.]
 [ 0. -1.  1.]
 [ 2.  0. -3.]]

A_inv (3 decimals) =
 [[ 0.273 -0.273  0.091]
 [ 0.182 -1.182 -0.273]
 [ 0.182 -0.182 -0.273]]


### 2c. Computing the Solution Vector

In [28]:
b_value = A_inv @ c
print("b (solution, 3 decimals) =\n", np.round(b_value, 3))

b (solution, 3 decimals) =
 [[-0.909]
 [-4.273]
 [ 0.727]]


### 2d. Overdetermined Systems

When there are more equations than unknowns, the system is **overdetermined** — there is generally no exact solution. The coefficient matrix is no longer square, so `np.linalg.inv` cannot be applied.

In [29]:
A_over = np.array([
    [ 1,  4,  7],
    [ 2, -5,  8],
    [ 3,  6, -9],
    [-1,  3,  0],
    [ 0,  1,  0]
], dtype=float)

c_over = np.array([3, 5, -4, 1, -2], dtype=float).reshape(5, 1)

print("A_over =\n", A_over)
print("\nc_over =\n", c_over)
print("\nshape of A_over =", A_over.shape)

A_over =
 [[ 1.  4.  7.]
 [ 2. -5.  8.]
 [ 3.  6. -9.]
 [-1.  3.  0.]
 [ 0.  1.  0.]]

c_over =
 [[ 3.]
 [ 5.]
 [-4.]
 [ 1.]
 [-2.]]

shape of A_over = (5, 3)


In [30]:
# np.linalg.inv requires a square matrix — this confirms the constraint
try:
    A_over_inv = np.linalg.inv(A_over)
except Exception as e:
    print("error when trying to invert A_over:", e)

error when trying to invert A_over: Last 2 dimensions of the array must be square


### 2e. The Moore-Penrose Pseudoinverse

For a non-square matrix $A$, the **Moore-Penrose pseudoinverse** $A^+$ generalises the concept of inversion. For full column-rank matrices it satisfies $A^+A = I$, which we verify below. NumPy computes it via SVD through `np.linalg.pinv`.

In [31]:
A_over_plus = np.linalg.pinv(A_over)
Aplus_A = A_over_plus @ A_over

print("A_plus @ A (3 decimals) =\n", np.round(Aplus_A, 3))

A_plus @ A (3 decimals) =
 [[ 1.  0. -0.]
 [ 0.  1. -0.]
 [ 0.  0.  1.]]


### 2f. Least-Squares Solution via Pseudoinverse

The pseudoinverse solution $\hat{\mathbf{b}} = A^+\mathbf{c}$ minimises the residual $\|\mathbf{c} - A\hat{\mathbf{b}}\|_2^2$ — the ordinary least-squares objective. It is the best approximation to an exact solution, but not exact when the system is overdetermined.

In [32]:
b_est = A_over_plus @ c_over

print("b estimate (3 decimals) =\n", np.round(b_est, 3))

residual = c_over - (A_over @ b_est)
print("\nresidual (c_over - A_over @ b_est) =\n", np.round(residual, 3))

b estimate (3 decimals) =
 [[ 0.168]
 [-0.066]
 [ 0.481]]

residual (c_over - A_over @ b_est) =
 [[-0.269]
 [ 0.485]
 [ 0.222]
 [ 1.367]
 [-1.934]]


The non-zero residual confirms there is no exact solution — the pseudoinverse gives the closest possible answer in a least-squares sense.

---
## 3. Machine Learning Paradigm Classification

Every task that involves learning from data falls into one of three paradigms — or none at all if the behaviour is entirely rule-based. The key questions are: (1) Are labelled examples available? (2) Is there an agent receiving feedback over time? (3) Is the logic fully hand-coded?

| # | Task | Paradigm | Reasoning |
|---|------|----------|-----------|
| a | Predicting house prices from features | **Supervised** | Labelled training pairs (features → price) |
| b | Clustering customers by purchase behaviour | **Unsupervised** | No labels; structure inferred from data |
| c | Robot navigating a maze via rewards/penalties | **Reinforcement Learning** | Agent optimises cumulative reward through interaction |
| d | Identifying even/odd numbers by remainder | **No learning** | Deterministic rule; no data or training |
| e | Labelling emails as spam using historical records | **Supervised** | Labelled dataset (spam / not spam) guides training |
| f | Grouping news articles without predefined labels | **Unsupervised** | No labels; topic structure discovered from content |
| g | Autonomous car trained via road/accident rewards | **Reinforcement Learning** | Policy learned through environmental feedback |
| h | Recommending movies from past ratings | **Supervised** | Historical ratings are labels for preference prediction |
| i | Grouping plant species by leaf patterns | **Unsupervised** | No class labels; similarity-based clustering |
| j | Robot learning chess via win/loss rewards | **Reinforcement Learning** | Agent maximises win reward over repeated episodes |
| k | Navigation app computing shortest path (Dijkstra) | **No learning** | Classical graph algorithm; no data-driven component |
| l | Predicting customer churn from account features | **Supervised** | Historical churn outcomes serve as labels |
| m | Segmenting an image without predefined labels | **Unsupervised** | Regions identified by visual similarity, not annotation |
| n | Converting Celsius to Fahrenheit via formula | **No learning** | Fixed mathematical formula; no training involved |
| o | Spam filter blocking emails by keyword rules | **No learning** | Hard-coded keyword matching; no learned model |
| p | Program learns to play a video game by score | **Reinforcement Learning** | Score is the reward signal; policy learned by trial and error |
| q | Clustering retail products by purchase patterns | **Unsupervised** | No predefined product categories; learned from co-purchase data |
| r | Diagnosing diseases from symptoms + diagnoses | **Supervised** | Ground-truth diagnoses are the training labels |
| s | Robotic arm trained to grasp objects | **Reinforcement Learning** | Reward only on successful grasp; sparse reward RL |
| t | Robot moving straight until obstacle, then turning | **No learning** | Preprogrammed reactive logic; no adaptation |

---
## 4. Regularisation and Cross-Validation

### 4a. Why λ* Cannot Be Learned from Training Data Alone

The regularised loss is:

$$\mathcal{L}(\mathbf{w}, \lambda) = \frac{1}{N_{\text{train}}} \sum_{i=1}^{N_{\text{train}}} (y_i - \hat{y}_i)^2 + \lambda \|\mathbf{w}\|_2^2$$

The penalty term $\lambda \|\mathbf{w}\|_2^2$ shrinks the weights toward zero. If $\lambda$ is selected by minimising $\mathcal{L}$ on the **training set**, the optimiser can always reduce the combined loss by decreasing $\lambda$ — a smaller penalty allows the model to fit the training data more tightly, which lowers the first term more than it raises the second. In the limit $\lambda \to 0$, the model interpolates the training data exactly, driving training loss to near zero while overfitting completely.

Because the training loss is a biased estimator of generalisation error, selecting $\lambda$ on training data systematically favours underfitting of the penalty. A separate **validation set** breaks this circularity: $\lambda^*$ is the value that minimises the **validation** loss, reflecting performance on data the model has not seen.

### 4b. Polynomial Feature Expansion: Term Count and Predictor Form

For $K$ input features and polynomial degree $d$, the number of terms in the expanded predictor is given by the stars-and-bars formula:

$$\binom{K + d}{d}$$

With $K = 2$ features $(x_1, x_2)$ and degree $d = 3$:

$$\binom{2 + 3}{3} = \binom{5}{3} = 10 \text{ terms}$$

Breaking these down by degree:

| Degree | Count | Terms |
|--------|-------|-------|
| 0 (constant) | 1 | $1$ |
| 1 (linear) | 2 | $x_1,\ x_2$ |
| 2 (quadratic) | 3 | $x_1^2,\ x_1 x_2,\ x_2^2$ |
| 3 (cubic) | 4 | $x_1^3,\ x_1^2 x_2,\ x_1 x_2^2,\ x_2^3$ |

The full predictor is:

$$\hat{y} = w_0 + w_1 x_1 + w_2 x_2 + w_3 x_1^2 + w_4 x_1 x_2 + w_5 x_2^2 + w_6 x_1^3 + w_7 x_1^2 x_2 + w_8 x_1 x_2^2 + w_9 x_2^3$$

**Cross-validation workflow:** $\lambda^*$ is selected by comparing validation loss across a grid of $\lambda$ values. Once chosen, the **final weights** $\mathbf{w}^*$ are computed by retraining on the full training set (train + validation combined) using $\lambda^*$ and the same regularised loss $\mathcal{L}(\mathbf{w}, \lambda^*)$.